# 3 Optimización de Hiperpárametros

## 3.01 Introduccion
En general los algoritmos que generan modelos predictivos poseen hiperparámetros que *dado un dataset* deben ser optimizados.
<br> La invocación de uno de esos algoritmos sin hiperparámetros no es más que
invocarlos con hiperparámetros por default definidos por el fabricante. Por ejemplo en el caso de la librería **rpart** los hiperparámetros por default son cp=0.01, maxdepth=30, minsplit=20, minbucket=6,  lo que en nuestro dataset genera un arbol de un solo nodo (decimos "no se abre el arbol"); la razon de esto es la proporcion de "BAJA+1" y "BAJA+2"

En el primer notebook de la asignatura usted probó optimizar manualmente los hiperparámetros entrenando en un mes completo y viendo los resultados directamente en el Public Leaderboard, que es una porción de los datos de futuro.
<br> En el mundo real no se dispone jamás de la clase del futuro, con lo cual lo anterior es meramente un artifical divertimento.
<br> La solución es estimar la bondad de un set de hiperparámetros en alguna combinación de:
  * Una sola partición de  <training, testing>
  * Multiples particiones de <training, testing>
  * El método de  k-fold Cross Validation , generalmente con n>=5
  * Utilizar   n-repated  k-fold Cross Validation
  * Leave One Out  si la cardinalidad del dataset y el poder de cómputo se lo permiten

Luego de comenzar a trabajar con el método de  "Multiples particiones de <training, testing>  se le invitó a extender un esqueleto de código del método de **Optimización de Hiperparámetros por Grid Search**



---



## 3.02 Conceptos

En esta entrega veremos los siguiente conceptos:
* *La maldición del ganador*, overfitting en los hiperparámetros ganadores, Selective Inference
* El origen del overfitting en un arbol de decisión
* Data Drifting
* Repensando el Overfitting, un árbol que se defiende solo del overfitting




---



## 3.03 Recapitulación

Si usted tuvo la oportunidad de encarar la Tarea para el Hogar 02 debería haber completado en la Planilla Colaborativa la hoja
* C3-Grid Search

con los resultados de su corrida de Grid Search

tenga presente que debe ordenar de mejor a peor y cargar en la planilla las posiciones 1, 2, 5, 10, 50, 100





---



## 3.04 El Rey (o reina) del Overfitting

A partir lo cargado en la planilla  **C3-Grid Search**
1. el profesor eligirá la combinación de hiperparámetros que generó la mayor ganancia de todo el curso y los cargará en las celdas naranjas de la hoja  **C3-GS-Overfitting**
2. utilizando la propia semilla, y esos "mejores hiperparámetros" cada alumno correrá el siguiente código y completará su correspondiente fila en la misma planilla
3. se discutirán en clase los resultados del experimento
4. si corresponde se relizarán nuevos experimentos con otras buenas combinaciones de hiperparámetros


#### Bibliografia

* Selective Inference - the silent killer of replicability   https://www.youtube.com/watch?v=6ZxIzVjV1DE
* Ioannidis, J. P. A. Why most published research findings are false. PLoS Med. 2, e124 (2005). https://journals.plos.org/plosmedicine/article/file?id=10.1371/journal.pmed.0020124&type=printable

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -s "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets



archivo_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/dataset_pequeno.csv"
archivo_destino="/content/datasets/dataset_pequeno.csv"
archivo_destino_bucket="/content/buckets/b1/datasets/dataset_pequeno.csv"

if ! test -f $archivo_destino_bucket; then
  wget  $archivo_origen  -O $archivo_destino_bucket
fi


if ! test -f $archivo_destino; then
  cp  $archivo_destino_bucket  $archivo_destino
fi

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if( !require("primes")) install.packages("primes")
require("primes")

Por favor cambiar:
* a SU semilla primigenia
* a los hiperparámetros ganadores de las celdas naranjas

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191
PARAM$qsemillas <- 11


PARAM$rpart <- list (
  "cp" = -0.5,
  "maxdepth" = 10,
  "minsplit" = 800,
  "minbucket" = 3
)

PARAM$training_pct <- 70L



In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

In [ ]:
ArbolEstimarGanancia <- function(semilla, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(param_basicos$training_pct, 100L -param_basicos$training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_basicos$rpart
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(list(
    "semilla" = semilla,
    "testing" = dataset[fold == 2, .N],
    "testing_pos" = dataset[fold == 2 & clase_ternaria == "BAJA+2", .N],
    "envios" = dataset[fold == 2, sum(prediccion[, "BAJA+2"] > 0.025)],
    "aciertos" = dataset[
        fold == 2,
        sum(prediccion[, "BAJA+2"] > 0.025 & clase_ternaria == "BAJA+2")
    ],
    "ganancia_test" = ganancia_test_normalizada
  ))
}

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp304"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# trabajo solo con los datos con clase, es decir 202107
dataset <- dataset[clase_ternaria != ""]

In [ ]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind= "L'Ecuyer-CMRG") # inicializo

# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )

In [ ]:

# la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
#  tantas veces como valores tenga el vector  PARAM$semillas
salidas <- mcmapply(ArbolEstimarGanancia,
  PARAM$semillas, # paso el vector de semillas
  MoreArgs = list(PARAM), # aqui paso el segundo parametro
  SIMPLIFY = FALSE,
  mc.cores = detectCores()
)

# muestro la tabla de salida
tb_salida <- rbindlist(salidas)
print( tb_salida)

In [ ]:
# finalmente calculo la media (promedio)  de las ganancias
cat( "ganancia: ", tb_salida[, mean(ganancia_test)], "\n" )

## 3.05 Comparando racionalmente modelos

El objetivo es determinar, utilizando el siguiente script, cuales son los **verdaderos** mejores hiperparámetros



In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")
require("ggplot2")

if (!require("primes")) install.packages("primes")
require("primes")

Cambiar por:
*  SU semilla primigenia
* en la tb_hiperparametros por los modelos que desee comparar

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191
PARAM$qsemillas <- 51


PARAM$tb_hiperparametros <- fread(
"cp,maxdepth,minsplit,minbucket,modelo
-0.5, 10, 800,3, martin01
-0.5, 8, 800,20, martin10
-0.5, 10, 2000,200, matias01
-1, 8, 2,500, gustavo1
-1, 8, 1000, 500, gustavo2
-1, 16, 2000, 500, gustavo3
")

PARAM$training_pct <- 70L

In [ ]:
PARAM$tb_hiperparametros

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

In [ ]:

ArbolEstimarGanancia <- function(semilla, training_pct, pcp, pmaxdepth, pminsplit, pminbucket) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division= c(training_pct, 100L -training_pct),
    agrupa= "clase_ternaria",
    seed= semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training
    xval= 0,
    cp= pcp,
    maxdepth= pmaxdepth,
    minsplit= pminsplit,
    minbucket= pminbucket
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(  ganancia_test_normalizada/ 1000000 )

}

In [ ]:
arboles <- function(pcp, pmaxdepth, pminsplit, pminbucket ) {

  mcmapply( ArbolEstimarGanancia,
    semilla= PARAM$semillas,
    PARAM$training_pct,
    pcp,
    pmaxdepth,
    pminsplit,
    pminbucket,
    mc.cores = detectCores()
  )
}

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp305"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# trabajo solo con los datos con clase, es decir 202107
dataset <- dataset[clase_ternaria != ""]

In [ ]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind= "L'Ecuyer-CMRG") # inicializo

# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )
print( PARAM$semillas )

In [ ]:
PARAM$tb_hiperparametros

In [ ]:
# Aqui hago el calculo pesado
tbl <- copy(PARAM$tb_hiperparametros )

tbl[, ganancias := Map( arboles, cp, maxdepth, minsplit, minbucket)]

In [ ]:
# calculo la media
tbl[, gan_media := unlist(Map( mean, ganancias)) ]
tbl


In [ ]:
# Expando los datos para poder graficar
tbl_ganancias <- tbl[, list(ganancia = unlist(ganancias)), by= modelo]

tbl_ganancias

In [ ]:
ggplot(tbl_ganancias, aes(x= modelo, y= ganancia)) +
  geom_boxplot() +
  stat_summary(
    fun = "median",
    geom = "text",
    aes(label = round(after_stat(y),1)),
    vjust = -0.5, # Adjust vertical position
    size = 3      # Adjust font size
  )

¿Cuál modelo elegiría?
¿Solo por la media?

**Test de Wilcoxon**

Utilizaremos el Test de Wilcoxon  https://en.wikipedia.org/wiki/Wilcoxon_signed-rank_test

In [ ]:
# genero una tabla donde computo el Test de Wilcoxon  de todos contra todos


tb_wilcox <- data.table(
  modelo1 = character(),
  modelo2 = character(),
  pvalue = numeric()
)

filas <-  nrow(tbl)

for( i in seq(filas-1) ) {
  for( j in seq(i+1, filas) ){

    res <- wilcox.test( tbl[i, ganancias[[1]] ],
      tbl[j, ganancias[[1]] ],
      paired= TRUE
    )

    tb_wilcox <- rbindlist( list(tb_wilcox,
      list(tbl[i,modelo], tbl[j, modelo], res$p.value)) )
  }
}


setorder( tb_wilcox, pvalue)
tb_wilcox

In [ ]:
# restrinjo a los Estadisticamente Significativos
tb_wilcox_valido <- tb_wilcox[ pvalue < 0.10 ]

# reordeno los nombres de tal forma que quede modelo1 < modelo2
for( i in seq(nrow(tb_wilcox_valido))) {
  reg <- as.list( tb_wilcox_valido[i])
  g1 <- tbl[ modelo== reg$modelo1, gan_media]
  g2 <- tbl[ modelo== reg$modelo2, gan_media]

  if( g1 > g2 )  {
    tb_wilcox_valido[i, `:=`(modelo1 = reg$modelo2, modelo2 = reg$modelo1)]
  }
}

tb_wilcox_valido
tb_wilcox_valido



---



## 3.06 Origen del Overfitting en un árbol de decisión
¿Qué combinacion de hiperparámetros overfitea un árbol de decisión, para nuestro dataset?
<br>¿Cómo se ve el overfitting desde el punto de vista de las curvas de ganancia?

El objetivo de este capítulo es que usted juegue manualmente con los hiperparámetros de un rpart, observe las curvas de ganancia generadas en una particion <training=50%, testing=50%>  y obtengla conclusiones sobre el fenómeno observado.

Introducimos el concepto de **Curva de Ganancia**
<br> Al aplicar un modelo a un dataset se le asigna a cada registro una probabilidad, a su vez cada registro contribuye con una ganancia la que puede ser una pérdida o una ganancia.  
<br>Ordenamos el dataset por probabilidad *descendente* y computamos la ganancia acumulada, generando de esta forma la curva de ganancia
<br> Para visualizar el efecto del under/over  fitting adecuadamente, realizamos una particion  <training= 50%, testing= 50%>



tener presente:
<br> Overfitting  **NO**  es la diferencia entre las curvas
<br> Lo que divide el underfitting del overfitting al aumentar la complejidad del modelo es la complejidad donde se alcanza la métrica máxima.

### ¿Qué debe hacer usted?
Probar al menos estas combinaciones:
* **Arbol crecimiento descontrolado**
   * cp= -1
   * maxdepth= 30
   * minsplit= 2
   * minbucket= 1
* Arbol talla reducida
   * cp= -1
   * maxdepth= 3
   * minsplit= 20000
   * minbucket= 10000


Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("ggplot2")

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp306"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

### Acción :  Jugar con  minsplit, minbucket y maxdepth

In [ ]:
# cambiar aqui los parametros
PARAM <- list()
PARAM$semilla_primigenia <- 102191


PARAM$minsplit <- 300
PARAM$minbucket <- 20
PARAM$maxdepth <- 11

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}


In [ ]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# a partir de ahora solo trabajo con 202107, el mes que tiene clase

dataset <- dataset[foto_mes == 202107] # defino donde voy a entrenar

In [ ]:
# La division training/testing es 50%, 50%
#  que sea 50/50 se indica con el c(1,1)

particionar(dataset,
  division= c(1, 1),
  agrupa= "clase_ternaria",
  seed= PARAM$semilla_primigenia
)

In [ ]:
# Entreno el modelo
# los datos donde voy a entrenar
# aqui es donde se deben probar distintos hiperparametros

modelo <- rpart(
  formula= "clase_ternaria ~ . -fold",
  data= dataset[fold == 1, ],
  xval= 0,
  cp= -1,
  minsplit= PARAM$minsplit,
  minbucket= PARAM$minbucket,
  maxdepth= PARAM$maxdepth
)

In [ ]:
# aplico el modelo a TODOS los datos, inclusive los de training
prediccion <- predict(modelo, dataset, type= "prob")

In [ ]:
# Pego la probabilidad de  BAJA+2
tb_prediccion <- dataset[, list(fold,clase_ternaria)]
tb_prediccion[, prob_baja2 := prediccion[, "BAJA+2"]]

In [ ]:
# Dibujo la curva de ganancia acumulada
setorder(tb_prediccion, fold, -prob_baja2)

In [ ]:
# agrego una columna que es la de las ganancias
# la multiplico por 2 para que ya este normalizada
#  es 2 porque cada fold es el 50%

tb_prediccion[, gan := 2 *ifelse(clase_ternaria == "BAJA+2", 0.975, -0.025)]
tb_prediccion[, ganancia_acumulada := cumsum(gan), by= fold]
tb_prediccion[, pos := sequence(.N), by= fold]

In [ ]:
tb_prediccion

In [ ]:
# agrego una columna que es la de las ganancias
# la multiplico por 2 para que ya este normalizada
#  es 2 porque cada fold es el 50%

tb_prediccion[, gan := 2 *ifelse(clase_ternaria == "BAJA+2", 0.975, -0.025)]
tb_prediccion[, ganancia_acumulada := cumsum(gan), by= fold]
tb_prediccion[, pos := sequence(.N), by= fold]

In [ ]:
# defino hasta donde muestra el grafico
amostrar <- 20000

In [ ]:
# Esta hermosa curva muestra como en el mentiroso training
#   la ganancia es siempre mejor que en el real testing

options( repr.plot.width=10, repr.plot.height=10)

gra <- ggplot(
           data= tb_prediccion[pos <= amostrar],
           aes( x= pos, y= ganancia_acumulada,
                color= ifelse(fold == 1, "train", "test") )
             ) + geom_line()

print( gra )


In [ ]:
# veo los resultados

print(PARAM)
cat( "Train gan max: ", tb_prediccion[fold==1, max(ganancia_acumulada)], "\n" )
cat( "Test  gan max: ", tb_prediccion[fold==2, max(ganancia_acumulada)], "\n" )




---



## 3.07  Corrida notebook inicial

En el repositorio oficial de la asignatura se encuentra el notebook ./src/arboles/z102_FinalTrain.ipynb  que automaticamente hace el submit a la Competencia Analista Sr  de Kaggle.
<br>  Ingrese a un nuevo Google Colab  y pruebe algunas corridas del notebook cambiando los hiperparámetros de rpart
<br> Porque:
* una cosa es ver la ganancia en testing
* y otra MUY DISTINTA verla en datos del futuro

## 3.05 Análisis de la salida de Grid Search

En clase utilizando un enfoque constructivista de educacion cada uno de los grupos del aula analizará las salidas de las corridas de Grid Search de la Tarea para el Hogar.
<br>Se espera que quienes ya trabajan como Data Analyst se luzcan en el análisis de esos datos
<br>Finalmente se utilizara un *arma conceptual secreta*, iluminando elegantemente donde están las mayores ganancias.

<br><br>Si usted no tuvo la oportunidad de hacer sus propias corridas esta generosa cátedra pone a su diposición esta salida https://storage.googleapis.com/open-courses/austral2025-af91/gridsearch.txt



---



## 3.07 Data Drifting  sospechas

Ordenar la salida del Grid Search en forma descendente por ganancia (en testing obviamente)
<br> De esta forma la posición 1 corresponde a la mayor ganancia, la 2 a la segunda mejor, etc
<br> En la Google Sheet Colaborativa,  hoja  **C3- GridSEarch** cargue las posiciones  1, 2, 5, 10, 50 y 100 de la salida del Grid Search, dejando la columna Public Leaderboard sin cargar
<br> La columna ganancia_mean tiene valores en orden descendente

El objetivo de hacer Grid Search  es utilizando particiones <training, testing>  encontrar los mejores hiperparámetros
<br> Esto tiene sentido en la medida que los hiperparámetros que resultan mejores de la búsqueda Grid Search son también los mejores cuando se hace el Final Training

Utilizando el notebook de la primiera clase,  **z102_FinalTrain.ipynb**   calcule para cada uno de los sets de hiperparámetros de las posiciones 1, 2, 5, 10, 50 y 100  cual es la ganancia en el Public Leaderboard de Kaggle
<br> Deberá hacer una corrida para cada conjunto de hiperparámetros

 ¿ Se cumple que los hiperparámetros de la posición  1 del Grid Search son los que mejor funcionan para predecir los datos del futuro ?

¿ Si esto no fuera así, estamos en una sitacion de **Game Over** ?



---



## 3.08 Data Drifting, breve intuicion

Se mostrará un posible origen de las discrepancias observadas en el capítulo anterior
<br> La solución al Data Drifting es otro precio ...

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp308"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
PARAM <- list()
PARAM$mes0 <- 202107
PARAM$mes1 <- 202109

In [ ]:
graficar_campo <- function(campo, param) {
  # quito de grafico las colas del 5% de las densidades
  qA <- quantile(dataset[foto_mes == param$mes0, get(campo)],
    prob= c(0.05, 0.95), na.rm= TRUE
  )

  qB <- quantile(dataset[foto_mes == param$mes1, get(campo)],
    prob= c(0.05, 0.95), na.rm= TRUE
  )

  xxmin <- pmin(qA[[1]], qB[[1]])
  xxmax <- pmax(qA[[2]], qB[[2]])

  densidad_A <- density(dataset[foto_mes == param$mes0, get(campo)],
    kernel= "gaussian", na.rm= TRUE
  )

  densidad_B <- density(dataset[foto_mes == param$mes1, get(campo)],
    kernel= "gaussian", na.rm= TRUE
  )

  plot(densidad_A,
    col= "blue",
    xlim= c(xxmin, xxmax),
    ylim= c(0, pmax(max(densidad_A$y), max(densidad_B$y))),
    main= campo
  )

  lines(densidad_B, col= "red", lty= 2)

  legend("topright",
    legend= c( param$mes0, param$mes1),
    col= c("blue", "red"), lty= c(1, 2)
  )
}


In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# Entreno el modelo
# utilizo los mejores hiperparametros encontrados

modelo <- rpart(
  formula= "clase_ternaria ~ . ",
  data= dataset[foto_mes == PARAM$mes0], # los datos donde voy a entrenar
  xval= 0,
  cp= -1,
  minsplit= 1144,
  minbucket= 539,
  maxdepth= 8
)


In [ ]:

campos_modelo <- names(modelo$variable.importance)
campos_buenos <- c(campos_modelo, setdiff(colnames(dataset), campos_modelo))
campos_buenos <- setdiff(
  campos_buenos,
  c("foto_mes", "clase_ternaria")
)

campos_buenos

In [ ]:
# para fines didacticos,  cliente_antiguedad primero
campos_buenos <- c("cliente_antiguedad", campos_buenos)

In [ ]:
# grafico las densidades de cada variable para los dos mses

options( repr.plot.width=15, repr.plot.height=15)

for (campo in campos_buenos) {
  cat(campo, "  ")
  graficar_campo(campo, PARAM)
}




---





---

